In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
import torch
import geopandas as gpd
import pandas as pd
import os
from pathlib import Path
import sys
def configure_proj_data():
    candidates = [
        Path(sys.prefix) / "Library" / "share" / "proj",
        Path(sys.prefix) / "Lib" / "site-packages" / "pyproj" / "proj_dir" / "share" / "proj",
        Path(os.environ.get("CONDA_PREFIX", "")) / "Library" / "share" / "proj",
        Path.home() / ".conda" / "envs" / "3s" / "Library" / "share" / "proj",
    ]

    for candidate in candidates:
        if (candidate / "proj.db").exists():
            os.environ["PROJ_LIB"] = str(candidate)
            os.environ["PROJ_DATA"] = str(candidate)

            import pyproj
            pyproj.datadir.set_data_dir(str(candidate))

            print("PROJ database:", candidate)
            return

    print("WARNING: proj.db not found")

configure_proj_data()

In [ ]:
feature_cols = ['elev_mean', 'slope_mean', 'ndvi_mean', 'water_pct',
       'developed_pct', 'barren_pct', 'forest_pct',
       'grassland_pct', 'pasture_hay_pct', 'cultivated_crops_pct',
       'woody_wetlands_pct', 'herbaceous_wetlands_pct',
       'poi_edu', 'poi_comm', 'poi_med', 'poi_trans', 'poi_pub', 'poi_food',
       'poi_finance', 'poi_diversity', 'minor_ratio',
       'elderly_ratio', 'sex_ratio', 'fam_inc_lt10k', 
       'fam_inc_10to15k', 'fam_inc_15to25k',
       'fam_inc_25to50k', 
       'fam_inc_50kplus',  'median_family_income',
       'persons_below_poverty', 'dist_water_m']

In [ ]:
dem_file = gpd.read_file(r"GIS_features/dem_features_15msa.csv")
print(dem_file)
ndvi_file = gpd.read_file(r"GIS_features/ndvi_features_15msa.csv")
print(ndvi_file)
nlcd_file = gpd.read_file(r"GIS_features/nlcd_features_15msa.csv")
print(nlcd_file)
poi_file = gpd.read_file(r"GIS_features/poi_features_15msa.csv")
print(poi_file)
pop_file = gpd.read_file(r"GIS_features/pop_features_15msa.csv")
print(pop_file)
eco_file = gpd.read_file(r"GIS_features/socioeconomic_features_15msa.csv")
print(eco_file)
tcc_file = gpd.read_file(r"GIS_features/tcc_features_15msa.csv")
print(tcc_file)
water_file = gpd.read_file(r"GIS_features/water_dist_features_15msa.csv")
print(water_file)
population = gpd.read_file(r"ACSDT5Y2020.pop/ACSDT5Y2020.B01003-Data.csv")
population = population.rename(columns={
    "GEO_ID": "cb_2020_3"
})

In [ ]:
gdf = dem_file.merge(ndvi_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(nlcd_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(poi_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(pop_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(eco_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(tcc_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(water_file,
                on="cb_2020_3",
                how="inner")
print(gdf)
gdf = gdf.merge(population,
                on="cb_2020_3",
                how="inner")
print(gdf)

In [ ]:
# MODIFIED START: 15 个城市 shp 的输入文件夹，以及输出 dataset 文件夹。
SHP_FILE = r"GIS_features/msa"
OUT_DIR = r"GIS_features/datasets"

os.makedirs(OUT_DIR, exist_ok=True)

print("SHP folder:", SHP_FILE)
print("Output folder:", OUT_DIR)
# MODIFIED END


In [ ]:
# MODIFIED START: 把单城市建图流程包装成函数，后面可以循环处理 15 个城市。
import re
import glob
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from libpysal.weights import Queen


def normalize_geoid_column(df):
    """
    统一 GEOID 字段名为 cb_2020_3。
    shp 里通常是 cb_2020__3，GIS 属性表里通常是 cb_2020_3。
    """
    if "cb_2020_3" in df.columns:
        pass
    elif "cb_2020__3" in df.columns:
        df = df.rename(columns={"cb_2020__3": "cb_2020_3"})
    elif "GEO_ID" in df.columns:
        df = df.rename(columns={"GEO_ID": "cb_2020_3"})
    else:
        raise ValueError("Cannot find GEOID column: cb_2020_3 / cb_2020__3 / GEO_ID")

    df["cb_2020_3"] = df["cb_2020_3"].astype(str).str.strip()
    return df


def city_name_from_shp(shp_path):
    """
    从 shp 文件名提取城市名，作为输出文件前缀。
    """
    stem = Path(shp_path).stem
    stem = stem.replace("2020_us_MSA_in_track_", "")
    stem = stem.replace("2020_MSA_in_track_", "")
    stem = stem.replace("_S", "")

    name_map = {
        "newyork": "newyork",
        "Knoxville": "Knoxvile",  # 保持和你前面实验里的城市命名一致
    }
    return name_map.get(stem, stem)


def make_city_dataset(shp_path, attribute_table, out_dir):
    """
    单个城市：
    1. 读取城市 tract shp；
    2. 按 cb_2020_3 连接 GIS 属性和 ACS 人口；
    3. 用 cb_2020_11 计算人口密度；
    4. 基于过滤后的 tract polygon 建 Queen graph；
    5. 输出 PyG Data 和 node_index csv。
    """
    city_name = city_name_from_shp(shp_path)
    print("\n==============================")
    print("Building city:", city_name)
    print("SHP:", shp_path)

    city_gdf = gpd.read_file(shp_path)
    city_gdf = normalize_geoid_column(city_gdf)

    attr = attribute_table.copy()
    attr = normalize_geoid_column(attr)

    before = len(city_gdf)
    overlap = len(set(city_gdf["cb_2020_3"]) & set(attr["cb_2020_3"]))

    # MODIFIED: 每个城市自己的 shp 和全量 GIS 属性表连接。
    # 注意这里保留 shp 的 geometry 和 cb_2020_11 面积字段。
    city_data = city_gdf.merge(
        attr,
        on="cb_2020_3",
        how="inner",
        suffixes=("", "_attr")
    )

    print("shp rows:", before)
    print("matched rows:", len(city_data))
    print("overlap GEOIDs:", overlap)

    if len(city_data) == 0:
        print("WARNING: no matched rows, skip", city_name)
        return None

    keep_cols = (
        ["cb_2020_3", "B01003_001E", "cb_2020_11", "geometry"]
        + feature_cols
    )

    missing_cols = [col for col in keep_cols if col not in city_data.columns]
    if missing_cols:
        raise ValueError(f"{city_name} missing columns: {missing_cols}")

    city_model = city_data[keep_cols].copy()

    for col in feature_cols:
        city_model[col] = pd.to_numeric(
            city_model[col],
            errors="coerce"
        )

    city_model["B01003_001E"] = pd.to_numeric(
        city_model["B01003_001E"],
        errors="coerce"
    )

    # MODIFIED: 用 Census ALAND 字段 cb_2020_11 算面积，单位 m2 -> km2。
    city_model["cb_2020_11"] = pd.to_numeric(
        city_model["cb_2020_11"],
        errors="coerce"
    )

    print("before drop:", len(city_model))

    city_model = city_model.replace([np.inf, -np.inf], np.nan)
    city_model = city_model.dropna(
        subset=feature_cols + ["B01003_001E", "cb_2020_11", "geometry"]
    )
    city_model = city_model[city_model["cb_2020_11"] > 0].copy()
    city_model = city_model.reset_index(drop=True)

    print("after drop:", len(city_model))

    if len(city_model) == 0:
        print("WARNING: all rows dropped, skip", city_name)
        return None

    # MODIFIED: dropna 后必须用 city_model 重新建 Queen graph，保证 edge_index 和 x/y 对齐。
    w = Queen.from_dataframe(
        city_model,
        use_index=False
    )
    print("nodes:", w.n)
    print("mean neighbors:", w.mean_neighbors)

    edges = []
    for i, neighbors in w.neighbors.items():
        for j in neighbors:
            edges.append([i, j])

    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(
            edges,
            dtype=torch.long
        ).t().contiguous()
        edge_index = to_undirected(edge_index)

    print("edge_index:", edge_index.shape)

    city_model["tract_area_sqkm"] = (
        city_model["cb_2020_11"] / 1_000_000
    )

    city_model["population_density"] = (
        city_model["B01003_001E"] / city_model["tract_area_sqkm"]
    )

    city_model["population_density"] = (
        city_model["population_density"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .clip(lower=0)
    )

    # MODIFIED: dataset 里保存 raw GIS features。
    # 后续训练/迁移时再只用 source training nodes fit StandardScaler，避免泄漏。
    x = torch.tensor(
        city_model[feature_cols].values,
        dtype=torch.float
    )

    y = torch.tensor(
        np.log1p(city_model["population_density"].values),
        dtype=torch.float
    )

    data = Data(
        x=x,
        edge_index=edge_index,
        y=y
    )

    data.city = city_name
    data.feature_names = [str(col) for col in feature_cols]  # MODIFIED: plain list for cross-version torch.load
    data.tract_id = city_model["cb_2020_3"].values

    data.population = torch.tensor(
        city_model["B01003_001E"].values,
        dtype=torch.float
    )

    data.tract_area_sqkm = torch.tensor(
        city_model["tract_area_sqkm"].values,
        dtype=torch.float
    )

    data.population_density = torch.tensor(
        city_model["population_density"].values,
        dtype=torch.float
    )

    node_info = city_model[
        [
            "cb_2020_3",
            "B01003_001E",
            "tract_area_sqkm",
            "population_density"
        ]
    ].copy()
    node_info["city"] = city_name

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pt_path = out_dir / f"{city_name}_gis_queen_graph.pt"
    csv_path = out_dir / f"{city_name}_node_index.csv"

    torch.save(data, pt_path)
    node_info.to_csv(csv_path, index=False)

    print("saved dataset:", pt_path)
    print("saved node index:", csv_path)
    print(data)

    return {
        "city": city_name,
        "shp": str(shp_path),
        "pt_path": str(pt_path),
        "node_index_path": str(csv_path),
        "nodes": data.num_nodes,
        "edges": data.num_edges,
        "matched_rows": len(city_data),
        "final_rows": len(city_model),
        "density_mean": float(city_model["population_density"].mean()),
        "density_median": float(city_model["population_density"].median()),
    }
# MODIFIED END


In [ ]:
# MODIFIED START: 循环处理 SHP_FILE 文件夹中的 15 个城市 shp，并输出最终 dataset。
shp_files = sorted(glob.glob(os.path.join(SHP_FILE, "*.shp")))

print("shp count:", len(shp_files))
for shp in shp_files:
    print(shp)

summary_records = []

for shp_path in shp_files:
    try:
        record = make_city_dataset(
            shp_path=shp_path,
            attribute_table=gdf,
            out_dir=OUT_DIR
        )
        if record is not None:
            summary_records.append(record)
    except Exception as e:
        print("FAILED:", shp_path)
        print(repr(e))

dataset_summary = pd.DataFrame(summary_records)
summary_path = os.path.join(OUT_DIR, "gis_dataset_summary.csv")
dataset_summary.to_csv(summary_path, index=False)

print("\n==============================")
print("Finished.")
print("Dataset summary saved:", summary_path)
display(dataset_summary)
# MODIFIED END
